# 0 · Read me first: the whole project in one page

**Goal.** Type in any coordinates on Earth and see **how much was built in one year**, from free satellite radar
alone. The model makes a land-cover map for every month (water, trees, grass & shrub, crops, built-up, bare & snow),
and the prototype measures how the built-up area changed.

**Why radar (Sentinel-1) and not normal photos (Sentinel-2)?** Optical satellites cannot see through clouds. In the
tropics (Lagos, Dhaka, the Amazon) some months have almost no cloud-free image, so monthly change cannot be measured.
Radar sends its own microwave signal and receives the echo: it works through clouds, at night, in every season. The
price: radar images do not look like photos, and some classes (grass vs crops vs bare soil) are hard to separate.

**The four notebooks**

| Notebook | Question it answers | Main files |
| --- | --- | --- |
| 01 Data | Where do the training examples come from, what do the input and the labels look like, how balanced and how noisy are they? | `data/*.py`, `data/tiles_v0/` |
| 02 Training & evaluation | Which network, which training choices, how good is it on places it never saw, which variant is best? | `train/*.py`, `train/runs/` |
| 03 Case studies | Does the prototype give sensible answers in real places, including hard ones? | `prototype/*.py`, `prototype/case_studies/` |
| 04 Training step by step | What happened during training, epoch by epoch and batch by batch, and how did the output maps change? | `train/runs/*/history.csv`, `steps.csv`, `snapshots/` |

**The pipeline in one picture**

```
 1 choose places            600 places of 2.56 km x 2.56 km, stratified over the world      data/sample_tiles.py
 2 download labels          monthly Dynamic World maps + Open Buildings roofs (2023)       data/export_tiles.py
 3 download radar           EVERY Sentinel-1 scene of 2023 over each place (~35,000)       data/export_scenes.py
 4 clean + align            6 classes, seasonal smoothing, everything on the same 10 m grid data/prepare_data.py
 5 balance                  each region / settlement type / place gets a fair share         train/balance.py
 6 train                    U-Net (ResNet34 encoder), one radar scene -> one map            train/train.py
 7 evaluate                 130 test places never seen in training, 11 variants compared   train/eval_*.py
 8 prototype                any point -> all scenes of 15 months -> 12 monthly maps         prototype/pipeline.py
                            -> class % and change -> built-up change -> checked against     prototype/app.py
                            two independent maps (ESRI land cover, Open Buildings Temporal)
```

**The most important decisions (and why)**

1. **Single raw scenes as training samples, not monthly averages.** Each radar scene is one example, so the
   model sees ~35,000 examples instead of 600 × 12 = 7,200, and it learns to cope with day-to-day radar noise. In the
   prototype, all scenes of a month are predicted and their probabilities averaged.
2. **Places stratified over the world.** If the training data were 20 German cities, the model would fail on African
   villages. Places were drawn by region × biome × settlement type × land cover, and the balanced sampler gives each
   region and settlement type an equal share of training.
3. **Spatial test split.** Test places lie in other 5° × 5° blocks than training places, so the test score is what a
   new, unseen place would get, not a neighbour of a training place.
4. **Hybrid, smoothed labels.** Dynamic World misses small houses and flickers between trees and shrub with the
   seasons; adding building footprints and smoothing over 3 months makes the labels cleaner (flicker 15.8 % → 9.8 %).
5. **6 classes, not 9.** Radar cannot tell grass from shrub; flooded vegetation and snow are rare and badly labelled.
6. **Ensemble + 3-month smoothing + a persistent-change rule + same-season comparison.** Each step removes false change:
   the month-to-month flicker of the maps fell from 25 % (one scene, one model) to 5.9 %.

**Headline results (130 unseen test places)**

| Setting | mIoU | Month-to-month flicker |
| --- | --- | --- |
| One model, one scene per month | 0.487 | 25 % |
| One model, mean of all scenes of the month | 0.522 | 15 % |
| Ensemble of 4 models, month mean | 0.544 | 12 % |
| **Ensemble + 3-month smoothing (prototype)** | **0.549** | **5.9 %** |

Built-up is the best class (F1 ≈ 0.84); grass & shrub vs crops vs bare are the hardest. Small village houses
are under-counted because a house of 8 m is smaller than one radar pixel.

## Glossary

**Radar and the input**

* **Sentinel-1**: two European radar satellites (C-band, wavelength 5.6 cm). In 2023 only Sentinel-1A was working
  (1B failed in Dec 2021), so a place is seen roughly every 6–12 days, from ascending and descending orbits. Where
  orbits overlap (much of Europe) there are many more scenes.
* **IW (Interferometric Wide swath)**: the standard mode over land, 250 km wide, 10 m pixels.
* **GRD**: Ground Range Detected, the processed image product (intensity only).
* **Backscatter (σ⁰, sigma-nought)**: how much of the radar signal comes back. Smooth surfaces (calm water, smooth
  sand, roads) reflect it away: dark. Rough surfaces and vegetation scatter some back: grey. Buildings act like corner
  reflectors (wall + ground, "double bounce"): very bright.
* **dB (decibel)**: backscatter is stored as 10·log10(σ⁰). Typical values: water −25 dB, forest −7 dB (VV),
  cities up to +5 dB.
* **VV, VH (polarisation)**: the radar sends a vertically polarised wave; VV = the vertical part of the echo, VH = the
  horizontal part. VH is created mainly by volume scattering (leaves, branches), so it is strong for vegetation.
  **VH − VV** (in dB = their ratio) is a third input channel that separates vegetation types.
* **Terrain flattening (γ⁰, gamma-nought)**: slopes facing the satellite look brighter than they are, slopes facing
  away look darker. Flattening divides by the illuminated area computed from a terrain model (Copernicus DEM, 30 m),
  so a forest on a slope looks like a forest on flat ground. Layover and shadow (steep terrain the radar cannot see)
  are masked.
* **Speckle**: grainy "salt and pepper" noise in every radar image, caused by interference between many small
  scatterers in one pixel. Reduced by averaging (several scenes, or a speckle filter such as the Lee filter).

**Labels and data**

* **Dynamic World**: Google / WRI land-cover maps from Sentinel-2 (optical), 9 classes, 10 m, for every cloud-free
  image. We take the most frequent class per pixel per month.
* **Open Buildings**: building footprints from high-resolution imagery (Africa, South & Southeast Asia, Latin
  America). Used to mark pixels with ≥ 15 % roof cover as built-up ("hybrid" label).
* **Tile / place**: a 256 × 256 pixel square (2.56 km × 2.56 km).
* **Scene / sample**: one radar acquisition over one place = one training example.
* **Train / validation / test split**: train = the network learns from it; validation = checked after every epoch
  to choose the best epoch; test = used once at the end, for the reported numbers.

**Network and training**

* **CNN (convolutional neural network)**: layers of small filters (3 × 3) that slide over the image and learn patterns:
  edges first, then textures, then objects.
* **Semantic segmentation**: a class for every pixel (not one label per image).
* **U-Net**: an encoder shrinks the image step by step (256 → 8 pixels) to understand context; a decoder enlarges it
  back to 256 to draw the map; skip connections pass fine details from the encoder to the decoder.
* **ResNet34**: the encoder; 34 layers with "residual" shortcuts that make deep networks trainable.
* **Transfer learning / ImageNet**: the encoder starts with weights learned on 1.2 million photos (ImageNet) instead
  of random numbers, then is fine-tuned on radar.
* **Logits → softmax → argmax**: the network outputs 6 scores per pixel (logits); softmax turns them into
  probabilities that sum to 1; argmax picks the class with the highest probability.
* **Loss**: the number training minimises. **Cross-entropy** punishes a low probability for the correct class;
  **Dice** measures the overlap of predicted and true areas per class (helps small classes).
* **Class weights**: rare classes count more in the loss so the network does not ignore them.
* **Epoch**: one pass through the training samples. **Batch**: 32 samples processed together; after each batch the
  weights are updated once (one **step**).
* **Optimizer (AdamW)** and **learning rate**: how big each weight update is. We start at 0.001 and lower it along a
  cosine curve.
* **Overfitting**: the network memorises the training places instead of learning general rules; training scores go
  up, validation scores stop improving. **Early stopping** ends training 10 epochs after the last improvement.
* **Augmentation**: random rotations and flips of training samples, so the network sees more variety.
* **Mixed precision (bf16)**: computing with 16-bit numbers where safe, about twice as fast on the GPU.

**Metrics**

* **TP / FP / FN**: true positive (correctly found), false positive (wrongly claimed), false negative (missed).
* **Overall accuracy (OA)**: share of pixels classified correctly. Can look good even if a rare class is never found.
* **Precision** = TP / (TP + FP): when the model says built-up, how often is it right?
* **Recall** = TP / (TP + FN): how much of the real built-up does it find?
* **F1** = 2 · precision · recall / (precision + recall).
* **IoU (intersection over union)** = TP / (TP + FP + FN): overlap of predicted and true area for one class.
  **mIoU** = the mean IoU over the 6 classes: the main score for segmentation (every class counts equally).
* **Confusion matrix**: rows = true class, columns = predicted class; the diagonal is correct, the rest shows which
  classes are mixed up.
* **Flicker / flip rate**: share of pixels whose class changes from one month to the next. Real land-use change is
  rare (< 1 % a year), so most flicker is error.

**Prototype**

* **Month mean**: the class probabilities of all scenes of a month averaged, then argmax.
* **Ensemble**: probabilities of 4 differently trained models averaged.
* **TTA (test-time augmentation)**: predict flipped copies too and average (tested, little gain).
* **smooth3**: each month's probabilities averaged with the previous and next month.
* **Same-season comparison**: built-up change = first 3 months vs the same 3 calendar months one year later.
* **Change rules**: *argmax* (built in most end months, in none of the start months), *moderate* and *confident*
  (built-up probability ≥ 0.5 at the end and below 0.35 / 0.2 in every start month).
* **Error level (noise floor)**: how much false new built-up the model reports on unseen test places per km² and year;
  a result is "clear growth" only at ≥ 3 × this level.

## How to read the notebooks

* Every section starts with a short title cell and is followed by **"How and why"** cells with this structure:
  **What** this step does · **Why** we did it this way · **How** it works (the code, in words) · **Reading the
  output** · **What we learned** (including what did not work).
* Code cells only read files that the scripts produced; notebooks 01, 02 and 04 run on the university server
  (where the data and the runs are), notebook 03 on any machine with the `prototype/case_studies/` files.
* Every attempt, what failed and why is also in the **Project Log** (Claude Docs).